# Tema Programare: Codificarea Trasaturilor Categoriale

Bun venit la tema de programare despre Codificarea Trasaturilor Categoriale!

Majoritatea seturilor de date din lumea reala contin variabile categoriale - categorii de produse, segmente de clienti, regiuni geografice. Modelele de machine learning au nevoie de intrari numerice, asa ca alegerea strategiei potrivite de codificare este critica: alegerea gresita poate introduce ordonari false, poate cauza probleme de memorie sau poate scurge in mod silentios informatii despre tinta.

**Ce vei face in aceasta tema:**

* **One-Hot Encoding** - Aplici OHE cu `pd.get_dummies` si `OneHotEncoder` si gestionezi cardinalitatea mare si categoriile nevazute
* **Ordinal Encoding** - Codifici trasaturi categoriale ordonate folosind o ordine de rang explicita si semnificativa
* **Frequency Encoding** - Inlocuiesti categoriile cu proportiile lor de frecventa din setul de antrenare
* **Target Encoding cu smoothing** - Codifici categoriile folosind statistici ale tintei, prevenind in acelasi timp scurgerea de informatie cu Bayesian smoothing
* **Pipeline de encoding** - Construiesti un pipeline complet `ColumnTransformer` care aplica encodari diferite pe coloane diferite
* **One-Hot Encoder de la zero** - Implementezi one-hot encoding folosind doar NumPy
* **Target Encoder de la zero** - Implementezi target encoding cu smoothing folosind doar NumPy si cautari in dictionar

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul. **Nu adauga si nu modifica cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, asa ca nu te baza pe celulele nou create pentru codul solutiei. Foloseste locurile oferite in notebook.

* Evita folosirea variabilelor globale daca nu este absolut necesar. Evaluatorul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Din acest motiv, variabilele globale pot sa nu fie disponibile la evaluare. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai apasand pe iconita 💾 din coltul stanga sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta sus.
---


## Cuprins

- [Exercitiul 1 - One-Hot Encoding](#exercise-1)
- [Exercitiul 2 - Ordinal Encoding](#exercise-2)
- [Exercitiul 3 - Frequency Encoding](#exercise-3)
- [Exercitiul 4 - Target Encoding cu smoothing](#exercise-4)
- [Exercitiul 5 - Pipeline de encoding](#exercise-5)
- [Exercitiul 6 - One-Hot Encoder de la zero](#exercise-6)
- [Exercitiul 7 - Target Encoder de la zero](#exercise-7)


## Importuri si configurare


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Set de date

Folosim un set de date sintetic de e-commerce / survey cu 500 de clienti. Fiecare rand reprezinta un client:

| Coloana | Tip | Descriere |
|:-------|:-----|:------------|
| `product_category` | Nominala | Tipul de produs cumparat |
| `education_level` | Ordinala | Cel mai inalt nivel de educatie atins |
| `satisfaction` | Ordinala | Scorul de satisfactie al clientului |
| `city` | Nominala (cardinalitate mare) | Orasul clientului (50 de valori unice) |
| `price` | Numerica | Pretul platit |
| `age` | Numerica | Varsta clientului |
| `churn` | Tinta binara | 1 daca clientul a plecat, 0 altfel |


In [ ]:
np.random.seed(42)
N = 500
CATEGORIES   = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports', 'Home']
EDUCATION    = ['High School', 'Bachelor', 'Master', 'PhD']
SATISFACTION = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']

df = pd.DataFrame({
    'product_category': np.random.choice(CATEGORIES, N),
    'education_level':  np.random.choice(EDUCATION, N, p=[0.30, 0.40, 0.20, 0.10]),
    'satisfaction':     np.random.choice(SATISFACTION, N, p=[0.05, 0.15, 0.35, 0.30, 0.15]),
    'city':             np.random.choice([f'City_{i:02d}' for i in range(50)], N),
    'price':            np.random.uniform(5.0, 500.0, N).round(2),
    'age':              np.random.randint(18, 70, N),
})

sat_weight = df['satisfaction'].map({'Poor':3,'Fair':2,'Good':1,'Very Good':0,'Excellent':-1})
edu_weight = df['education_level'].map({'High School':1,'Bachelor':0.5,'Master':0.0,'PhD':-0.5})
logit = -0.5 + 0.4*sat_weight + 0.2*edu_weight + 0.001*df['price'] + np.random.normal(0,1,N)
df['churn'] = (logit > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('churn', axis=1), df['churn'], test_size=0.2, random_state=42)

print(f'Dataset shape: {df.shape}')
print(f'Churn rate:    {df["churn"].mean():.2f}')
print(f'\nFirst 3 rows:')
df.head(3)

<a id='exercise-1'></a>
## Exercitiul 1 - One-Hot Encoding

### Context

**One-Hot Encoding (OHE)** este tehnica standard pentru trasaturi categoriale **nominale** (neordonate).  
Fiecare categorie unica devine propria coloana binara: `1` cand categoria este prezenta, `0` in caz contrar.

| Inainte de OHE | Dupa OHE |
|:-----------|:----------|
| `product_category = 'Food'` | `category_Food = 1`, toate celelalte = 0 |

**Cand se foloseste:** modele liniare, regresie logistica, SVM-uri - orice model care presupune intrari numerice fara ordonare implicita.

### Sarcina

Implementeaza `apply_one_hot_encoding(df, columns)` care:
1. Aplica OHE pe fiecare coloana listata in `columns`
2. **Elimina** coloanele categoriale originale
3. Returneaza DataFrame-ul modificat

**Indicii:**
- `pd.get_dummies(df, columns=columns, dtype=int)` face toate acestea intr-un singur apel.
- Asigura-te ca valorile coloanelor din iesire sunt intregi (0 sau 1), nu booleeni.


In [ ]:
def apply_one_hot_encoding(df, columns):
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    encoder = None       # OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    X_enc   = None       # fit_transform on df[columns]
    cols    = None       # encoder.get_feature_names_out(columns)
    df_enc  = None       # pd.DataFrame(X_enc, columns=cols, index=df.index)
    return pd.concat([df.drop(columns=columns), df_enc], axis=1)
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
# Quick sanity check
df_ohe = apply_one_hot_encoding(df.drop('churn', axis=1), ['product_category'])
new_cols = [c for c in df_ohe.columns if c.startswith('product_category_')]
print(f'Original columns: {list(df.columns)}')
print(f'New OHE columns:  {new_cols}')
print(f'Rows sum to 1:    {df_ohe[new_cols].sum(axis=1).eq(1).all()}')
df_ohe.head(3)

In [ ]:
unittests.exercise_1(apply_one_hot_encoding)

<a id='exercise-2'></a>
## Exercitiul 2 - Ordinal Encoding

### Context

**Ordinal encoding** mapeaza fiecare categorie la un numar intreg care pastreaza **ordinea** semnificativa a categoriilor.  
Este adecvat doar pentru trasaturi **ordinale** (ordonate):

| Satisfactie | Valoare ordinala |
|:-------------|:--------------|
| Poor         | 0             |
| Fair         | 1             |
| Good         | 2             |
| Very Good    | 3             |
| Excellent    | 4             |

> ⚠️ **Nu** aplica niciodata ordinal encoding pe trasaturi nominale - introduce informatie falsa de ordonare.

### Sarcina

Implementeaza `apply_ordinal_encoding(df, col, order)` care:
1. Mapeaza fiecare valoare din `df[col]` la pozitia sa intreaga in lista `order` (indexata de la 0)
2. Adauga rezultatul ca o coloana noua numita `col + '_encoded'`
3. Pastreaza coloana originala neschimbata
4. Returneaza DataFrame-ul modificat

**Indicii:**
- Construieste un `mapping = {cat: i for i, cat in enumerate(order)}`
- Foloseste `Series.map(mapping)` pentru a aplica maparea.


In [ ]:
def apply_ordinal_encoding(df, col, order):
    ### ÎNCEPUT DE COD AICI ### (≈ 4 lines)
    encoder = None       # OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1)
    df = df.copy()
    df[col] = None       # encoder.fit_transform(df[[col]])
    return df
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
EDUCATION_ORDER = ['High School', 'Bachelor', 'Master', 'PhD']
SATISFACTION_ORDER = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']

df_ord = apply_ordinal_encoding(df, 'satisfaction', SATISFACTION_ORDER)
print('Ordinal mapping verification:')
print(df_ord[['satisfaction', 'satisfaction_encoded']].drop_duplicates().sort_values('satisfaction_encoded').to_string())

In [ ]:
unittests.exercise_2(apply_ordinal_encoding)

<a id='exercise-3'></a>
## Exercitiul 3 - Frequency Encoding

### Context

**Frequency encoding** inlocuieste fiecare categorie cu **proportia** de randuri care contin acea categorie in datele de antrenare.

| Oras    | Numar | Frecventa |
|:--------|:------|:----------|
| NYC     | 12    | 12/50 = 0.24 |
| LA      | 8     | 8/50 = 0.16 |
| Chicago | 3     | 3/50 = 0.06 |

Produce o **singura coloana numerica** indiferent de cardinalitate - ideala pentru trasaturi cu cardinalitate mare si pentru modele bazate pe arbori.

> ⚠️ Calculeaza frecventele doar pe datele de antrenare - niciodata pe intregul set de date care include si randurile de test.

### Sarcina

Implementeaza `apply_frequency_encoding(df, col)` care:
1. Calculeaza proportia fiecarei categorii folosind `value_counts(normalize=True)`
2. Adauga o coloana noua `col + '_freq'` cu valoarea frecventei pentru fiecare rand
3. Pastreaza coloana originala neschimbata
4. Returneaza DataFrame-ul modificat

**Indicii:**
- `df[col].value_counts(normalize=True)` returneaza un Series de proportii.
- Foloseste `.map()` pentru a atribui fiecarui rand proportia categoriei sale.


In [ ]:
def apply_frequency_encoding(df, col):
    ### ÎNCEPUT DE COD AICI ### (≈ 4 lines)
    freq_map = None      # pandas value_counts(normalize=True).to_dict()
    df = df.copy()
    df[col + '_freq'] = None   # map using freq_map
    return df
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
df_freq = apply_frequency_encoding(df, 'city')
print('Top 5 cities by frequency:')
print(df_freq[['city', 'city_freq']].drop_duplicates().sort_values('city_freq', ascending=False).head().to_string())

In [ ]:
unittests.exercise_3(apply_frequency_encoding)

<a id='exercise-4'></a>
## Exercitiul 4 - Target Encoding cu smoothing

### Context

**Target encoding** inlocuieste fiecare categorie cu **media valorii tintei** pentru randurile din acea categorie.
Mediile brute sunt zgomotoase pentru categoriile rare (de exemplu, un oras vazut o singura data in antrenare).
**Smoothing-ul** trage estimarile pentru categoriile rare catre media globala:

$$\text{encoded}(c) = \frac{n_c \cdot \bar{y}_c + \alpha \cdot \bar{y}_{\text{global}}}{n_c + \alpha}$$

unde $n_c$ = numarul de randuri cu categoria $c$, $\bar{y}_c$ = media categoriei, $\alpha$ = intensitatea smoothing-ului.

> ⚠️ Calculeaza intotdeauna statisticile pentru target encoding **doar din setul de antrenare** pentru a preveni scurgerea de informatie despre tinta.

### Sarcina

Implementeaza `apply_target_encoding(X_train, y_train, X_test, col, alpha=10)` care:
1. Calculeaza media globala si valorile per-categorie (media, count) din `X_train` + `y_train`
2. Aplica formula de smoothing pentru a obtine encoding-ul `smoothed` pentru fiecare categorie
3. Adauga `col + '_target_enc'` in copii ale lui `X_train` si `X_test`
4. Pentru categoriile necunoscute din `X_test`, foloseste media globala
5. Returneaza `(X_train_enc, X_test_enc)`

**Indicii:**
- Foloseste `pd.DataFrame({'cat': X_train[col], 'y': y_train}).groupby('cat')['y'].agg(['mean','count'])`
- Aplica `.fillna(global_mean)` pe coloana mapata din test pentru a gestiona categoriile necunoscute.


In [ ]:
def apply_target_encoding(X_train, y_train, X_test, col, alpha=10):
    ### ÎNCEPUT DE COD AICI ### (≈ 7 lines)
    global_mean = None        # np.mean(y_train)
    cat_means   = {}          # dict: category → mean target on training rows
    smoothed    = {}          # apply smoothing formula per category
    X_train = X_train.copy()
    X_test  = X_test.copy()
    X_train[col + '_te'] = None   # map training col using smoothed dict
    X_test[col  + '_te'] = None   # map test col (unseen → global_mean)
    return X_train, X_test
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
X_tr_enc, X_te_enc = apply_target_encoding(X_train, y_train, X_test, 'city', alpha=10)
print('City target encoding — train sample:')
print(X_tr_enc[['city', 'city_target_enc']].drop_duplicates().sort_values('city_target_enc').head(8).to_string())

In [ ]:
unittests.exercise_4(apply_target_encoding)

<a id='exercise-5'></a>
## Exercitiul 5 - Pipeline de encoding

### Context

O strategie de encoding pregatita pentru productie foloseste `ColumnTransformer` din sklearn pentru a aplica **encodari diferite pe coloane diferite** simultan, apoi trimite iesirea combinata catre un clasificator.

```
Input DataFrame
  │
  ├─ nominal_cols  ──► OneHotEncoder  ──┐
  ├─ ordinal_col   ──► OrdinalEncoder ──┼──► combined ──► LogisticRegression
  └─ 'price'       ──► StandardScaler ──┘
```

Obiectul `Pipeline` se asigura ca encoder-ele fac **fit doar pe datele de antrenare** in timpul cross-validarii, prevenind scurgerea de informatie.

### Sarcina

Implementeaza `create_encoding_pipeline(nominal_cols, ordinal_col, ordinal_order)` care:
1. Creeaza un `ColumnTransformer` cu trei transformatori:
   - `'ohe'`: `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` pe `nominal_cols`
   - `'ord'`: `OrdinalEncoder(categories=[ordinal_order])` pe `[ordinal_col]`
   - `'num'`: `StandardScaler()` pe `['price']`
2. Returneaza un `Pipeline([('ct', column_transformer), ('clf', LogisticRegression(max_iter=500))])`

**Indicii:**
- `ColumnTransformer` primeste o lista de tuple `(name, transformer, columns)`.
- Pune `ordinal_col` intr-o lista: `[ordinal_col]`.


In [ ]:
def create_encoding_pipeline(nominal_cols, ordinal_col, ordinal_order):
    ### ÎNCEPUT DE COD AICI ### (≈ 6 lines)
    ohe = None       # OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    oe  = None       # OrdinalEncoder(categories=[ordinal_order])
    ct  = None       # ColumnTransformer with ohe for nominal_cols, oe for [ordinal_col], remainder='passthrough'
    return Pipeline([('preprocessing', ct), ('clf', LogisticRegression(max_iter=1000))])
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
EDUCATION_ORDER = ['High School', 'Bachelor', 'Master', 'PhD']

pipeline = create_encoding_pipeline(
    nominal_cols=['product_category'],
    ordinal_col='education_level',
    ordinal_order=EDUCATION_ORDER,
)

X_pipe = df[['product_category', 'education_level', 'price']]
y_pipe = df['churn']

from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline, X_pipe, y_pipe, cv=5, scoring='accuracy')
print(f'5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
unittests.exercise_5(create_encoding_pipeline)

<a id='exercise-6'></a>
## Exercitiul 6 - One-Hot Encoder de la zero

### Context

Intelegerea modului in care functioneaza OHE in interior te ajuta sa depanezi cazuri limita si sa il adaptezi la cerinte personalizate.
Algoritmul este:

1. Colectezi toate categoriile unice si le sortezi.
2. Creezi o matrice de zerouri cu forma `(n_samples, n_unique)`.
3. Pentru fiecare rand `i`, setezi la 1 coloana corespunzatoare lui `series[i]`.

### Sarcina

Implementeaza `one_hot_encode_from_scratch(series)` care:
1. Colecteaza toate valorile **unice** din Series si le sorteaza
2. Creeaza o **matrice intreaga de zerouri** cu forma `(len(series), n_unique)`
3. Pentru fiecare rand, seteaza coloana corecta la 1
4. Returneaza `(matrix, categories)` unde `categories` este lista sortata a valorilor unice

**Indicii:**
- `sorted(series.unique())` iti da categoriile sortate.
- Construieste un dictionar `cat_to_idx = {c: i for i, c in enumerate(categories)}` pentru cautare rapida.
- Foloseste `np.zeros((n, k), dtype=int)` pentru a crea matricea.


In [ ]:
def one_hot_encode_from_scratch(series):
    ### ÎNCEPUT DE COD AICI ### (≈ 6 lines)
    categories = None     # sorted unique values from series
    result     = {}       # build a dict: category → 0/1 array
    for cat in categories:
        result[cat] = None   # (series.values == cat).astype(int)
    return pd.DataFrame(result, index=series.index)
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
demo_series = pd.Series(['Food', 'Electronics', 'Food', 'Books', 'Electronics'])
matrix, cats = one_hot_encode_from_scratch(demo_series)
print(f'Categories (sorted): {cats}')
print(f'Matrix shape:        {matrix.shape}')
print(f'Matrix:\n{matrix}')
print(f'Row sums:            {matrix.sum(axis=1).tolist()}')

In [ ]:
unittests.exercise_6(one_hot_encode_from_scratch)

<a id='exercise-7'></a>
## Exercitiul 7 - Target Encoder de la zero

### Context

Acum vei implementa **target encoding cu smoothing** de la primele principii folosind doar NumPy si pandas - fara sklearn.

Reamintire a formulei de smoothing:

$$\text{encoded}(c) = \frac{n_c \cdot \bar{y}_c + \alpha \cdot \bar{y}_{\text{global}}}{n_c + \alpha}$$

Pentru **categoriile necunoscute** la testare (categorii nevazute in antrenare), revino la media globala.

### Sarcina

Implementeaza `target_encode_from_scratch(train_col, y_train, test_col, alpha=10)` care:
1. Calculeaza media globala a lui `y_train`
2. Calculeaza media si count per-categorie din `train_col` si `y_train`
3. Aplica formula de smoothing pentru a obtine maparea fiecarei categorii
4. Codifica `train_col` si `test_col` folosind aceasta mapare
5. Foloseste media globala pentru categoriile din `test_col` care nu au fost vazute la antrenare
6. Returneaza `(train_encoded, test_encoded)` ca `np.ndarray`

**Indicii:**
- Foloseste `pd.DataFrame({'cat': train_col, 'y': y_train}).groupby('cat')['y'].agg(['mean','count'])`
- Foloseste `mapping.get(c, global_mean)` pentru a gestiona categoriile necunoscute.


In [ ]:
def target_encode_from_scratch(train_col, y_train, test_col, alpha=10):
    ### ÎNCEPUT DE COD AICI ### (≈ 8 lines)
    global_mean = None    # np.mean(y_train)
    enc_map     = {}      # category → smoothed mean
    for cat in train_col.unique():
        mask      = None  # train_col == cat
        n         = None  # mask.sum()
        cat_mean  = None  # y_train[mask].mean()
        enc_map[cat] = None  # smoothing formula: (n*cat_mean + alpha*global_mean) / (n + alpha)
    train_enc = None  # train_col.map(enc_map).fillna(global_mean)
    test_enc  = None  # test_col.map(enc_map).fillna(global_mean)
    return train_enc, test_enc
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
train_cities = X_train['city'].reset_index(drop=True)
test_cities  = X_test['city'].reset_index(drop=True)

tr_enc, te_enc = target_encode_from_scratch(
    train_cities, y_train.values, test_cities, alpha=10
)
print(f'Train encoding — first 5: {tr_enc[:5].round(3).tolist()}')
print(f'Test  encoding — first 5: {te_enc[:5].round(3).tolist()}')
print(f'Global mean: {y_train.mean():.4f}')

In [ ]:
unittests.exercise_7(target_encode_from_scratch)